# Supplementary File S6 — Per-Class Precision, Recall, and F1-Score (100 categories, test set)

This notebook derives the complete per-class evaluation table referenced in Section 5.3 of the paper ("The complete 100 x 100 confusion matrix and per-class precision, recall, and F1-score are provided as Supplementary File S6"). It auto-locates `test_predictions.csv` (produced by S1) inside your `CEP_Food_Dataset` Drive folder and writes a single clean CSV table as the standalone artifact.

**Note:** run this after S1 has produced `test_predictions.csv` from the trained model's test-set predictions - the table is only meaningful once populated with real evaluation output, not placeholder data. No GPU is needed; the image cache is pulled only if the predictions store integer labels that need mapping to dish names.

---
## Step 0: Mount Drive and Load Predictions

Everything reads from your `CEP_Food_Dataset` Drive folder — no manual uploads needed. If the
predictions store integer class indices, they are mapped back to dish names using the **same**
class order S1 used (sorted folder names from the `resized_256` cache).

In [2]:
import os, glob, zipfile
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT  = "/content/drive/MyDrive/CEP_Food_Dataset"
RESIZED_DIR = "/content/resized_256"
OUTPUT_PATH = f"{DRIVE_ROOT}/S6_per_class_precision_recall_f1.csv"

# Locate the predictions file anywhere under the project folder (newest match wins).
csv_matches = glob.glob(f"{DRIVE_ROOT}/**/test_predictions.csv", recursive=True)
assert csv_matches, (
    "test_predictions.csv not found under " + DRIVE_ROOT +
    ". Run S1 to the end - it writes this file from the trained model's test predictions.")
TEST_PREDICTIONS_PATH = sorted(csv_matches, key=os.path.getmtime)[-1]
preds_df = pd.read_csv(TEST_PREDICTIONS_PATH)

def _has_class_dirs(d):
    return os.path.isdir(d) and any(
        os.path.isdir(os.path.join(d, x)) for x in os.listdir(d))

# If labels are stored as integer indices, map them to dish names using the SAME
# class order S1 used (sorted folder names). Extract the cache only if needed.
label_cols = [c for c in ['true_class', 'pred_class'] if c in preds_df.columns]
needs_mapping = all(pd.api.types.is_integer_dtype(preds_df[c]) for c in label_cols)

if needs_mapping:
    if not _has_class_dirs(RESIZED_DIR):
        zip_matches = glob.glob(f"{DRIVE_ROOT}/**/resized_256.zip", recursive=True)
        assert zip_matches, (
            "Labels are integer indices but resized_256.zip was not found under " +
            DRIVE_ROOT + " to recover the class names. Run S1's preprocessing cell first.")
        with zipfile.ZipFile(sorted(zip_matches, key=os.path.getmtime)[-1]) as z:
            z.extractall(RESIZED_DIR)
    class_names = sorted(
        d for d in os.listdir(RESIZED_DIR)
        if os.path.isdir(os.path.join(RESIZED_DIR, d)))
    assert len(class_names) == 100, f"Expected 100 class folders, found {len(class_names)}."
    idx_to_class = {i: name for i, name in enumerate(class_names)}
    preds_df['true_class'] = preds_df['true_class'].map(idx_to_class)
    preds_df['pred_class'] = preds_df['pred_class'].map(idx_to_class)

print(f"Predictions file: {TEST_PREDICTIONS_PATH}")
print(f"Loaded {len(preds_df)} test predictions across "
      f"{preds_df['true_class'].nunique()} classes.")
preds_df.head()

Mounted at /content/drive
Predictions file: /content/drive/MyDrive/CEP_Food_Dataset/outputs/test_predictions.csv
Loaded 687 test predictions across 100 classes.


,true_label_idx,pred_label_idx,true_class,pred_class,confidence
0,92,92,Upma,Upma,0.994346
1,40,40,Jamun_shrikhand,Jamun_shrikhand,0.992244
2,0,0,Aalubhat,Aalubhat,0.988402
3,20,16,Chicken_tandoor,Chicken_crispy,0.750082
4,60,70,Misal_pav,Pav_bhaji,0.911872


---
## Step 1: Build the Per-Class Table

`classification_report` gives precision, recall, F1, and support for every class in one pass.
Rows are sorted by recall so the hardest (most-confused) dishes surface at the top.

In [3]:
report_dict = classification_report(
    preds_df['true_class'], preds_df['pred_class'],
    output_dict=True, digits=4, zero_division=0
)

report_df = pd.DataFrame(report_dict).transpose()
report_df.index.name = 'class'
report_df = report_df.rename(columns={'f1-score': 'f1_score'})

# Separate the per-class rows from the summary rows (accuracy / macro avg / weighted avg)
summary_rows = report_df.loc[['accuracy', 'macro avg', 'weighted avg']] if 'accuracy' in report_df.index else None
per_class_df = report_df.drop(index=['accuracy', 'macro avg', 'weighted avg'], errors='ignore')
per_class_df = per_class_df.sort_values('recall')

print(f"Per-class table: {len(per_class_df)} rows")
per_class_df.head(10)

Per-class table: 100 rows


,precision,recall,f1_score,support
class,,,,
Veg_sandwich,1.000000,0.333333,0.500000,3.0
Dahi_Kadhi,1.000000,0.500000,0.666667,2.0
Manchow_soup,1.000000,0.500000,0.666667,2.0
Chilli_chicken,0.666667,0.500000,0.571429,4.0
Misal_pav,0.666667,0.500000,0.571429,4.0
Fish_fry,1.000000,0.600000,0.750000,5.0
Mango_barfi,0.714286,0.625000,0.666667,8.0
Chicken_fry,0.666667,0.666667,0.666667,6.0
Sabudana_Usal,1.000000,0.666667,0.800000,3.0


---
## Step 2: Save the Standalone S6 Artifact

In [4]:
per_class_df.to_csv(OUTPUT_PATH)
print(f"Saved: {OUTPUT_PATH}")

if summary_rows is not None:
    print("\nSummary metrics (macro / weighted / accuracy):")
    print(summary_rows)

Saved: /content/drive/MyDrive/CEP_Food_Dataset/S6_per_class_precision_recall_f1.csv

Summary metrics (macro / weighted / accuracy):
              precision    recall  f1_score     support
class                                                  
accuracy       0.906841  0.906841  0.906841    0.906841
macro avg      0.920188  0.896596  0.897766  687.000000
weighted avg   0.921575  0.906841  0.907053  687.000000


## Sanity Checks

Before uploading this file to the repository, confirm:
- All 100 classes are present as rows (no class silently dropped due to zero predicted samples).
- `support` column sums to the total test-set size.
- No `NaN` values in `precision` / `recall` / `f1_score` (these appear if a class had zero
  predicted OR zero true samples — `zero_division=0` above sets these to 0 rather than NaN,
  but it's worth checking which classes triggered it).

In [5]:
assert len(per_class_df) == preds_df['true_class'].nunique(), \
    "Row count mismatch -- a class may be missing from the report."

support_total = per_class_df['support'].sum()
print(f"Support column sums to: {support_total} (should equal test-set size: {len(preds_df)})")

zero_precision_classes = per_class_df[per_class_df['precision'] == 0]
if len(zero_precision_classes) > 0:
    print(f"\n{len(zero_precision_classes)} class(es) with zero precision (model never "
          f"predicted this class correctly, or never predicted it at all):")
    print(zero_precision_classes.index.tolist())
else:
    print("\nNo classes with zero precision.")

Support column sums to: 687.0 (should equal test-set size: 687)

No classes with zero precision.
